In [ ]:
%%writefile para_mod.py
import numpy as np

device = "CPU"
num_cores = 4
Scheme = "EULER"

# Domain
Lx = 1.0
Lz = 1.0
Nx = 120   # you can modify resolution here
Nz = 120

# Time
tinit = 0.0
tfinal = 0.01
CFL = 0.5   # use CFL, not fixed dt

# Gas properties
gamma = 1.4
R = 287.0

# Initial flow
Mach0 = 2.0
T0 = 300.0
p0 = 101325.0

# Output
t_p = 0.002
output_dir = "output_mod"


In [ ]:
%%writefile grid_mod.py
import numpy as np
import para_mod as para

ncp = np
copy = np.copy

# --------------------------------
# Grid spacing
# --------------------------------
dx = para.Lx / para.Nx
dz = para.Lz / para.Nz

# --------------------------------
# Cell-centered coordinates
# --------------------------------
X = (np.arange(para.Nx) + 0.5) * dx
Z = (np.arange(para.Nz) + 0.5) * dz

X_mesh, Z_mesh = np.meshgrid(X, Z, indexing='ij')


In [ ]:
%%writefile divergence_mod.py
import numpy as np
import grid_mod as grid
from riemann_mod import rusanov_flux_x, rusanov_flux_z

def compute_flux_divergence(Q):
    """
    Computes:
    div(F) = (F_{i+1/2} - F_{i-1/2})/dx
           + (G_{j+1/2} - G_{j-1/2})/dz
    """

    # -----------------------------
    # X-direction face fluxes
    # -----------------------------
    QL = Q[:, :-1, :]
    QR = Q[:, 1:, :]
    Fx = rusanov_flux_x(QL, QR)

    div_x = np.zeros_like(Q)
    div_x[:, 1:-1, :] = (Fx[:, 1:, :] - Fx[:, :-1, :]) / grid.dx

    # -----------------------------
    # Z-direction face fluxes
    # -----------------------------
    QLz = Q[:, :, :-1]
    QRz = Q[:, :, 1:]
    Fz = rusanov_flux_z(QLz, QRz)

    div_z = np.zeros_like(Q)
    div_z[:, :, 1:-1] = (Fz[:, :, 1:] - Fz[:, :, :-1]) / grid.dz

    return div_x + div_z


In [ ]:
%%writefile compressible_mod.py
import numpy as np
import para_mod as para
from divergence_mod import compute_flux_divergence

# ----------------------------
# Primitive variables
# ----------------------------
rho = np.zeros((para.Nx, para.Nz))
ux  = np.zeros((para.Nx, para.Nz))
uz  = np.zeros((para.Nx, para.Nz))
p   = np.zeros((para.Nx, para.Nz))

# ----------------------------
# Conserved variables
# Q = [rho, rho*u, rho*v, E]
# ----------------------------
Q = np.zeros((4, para.Nx, para.Nz))

# -------------------------------------------------
# Update conserved variables from primitive
# -------------------------------------------------
def update_conserved():
    Q[0] = rho
    Q[1] = rho * ux
    Q[2] = rho * uz
    Q[3] = p/(para.gamma - 1) + 0.5*rho*(ux**2 + uz**2)

# -------------------------------------------------
# Update primitive variables from conserved
# -------------------------------------------------
def update_primitive():
    rho[:] = Q[0]
    ux[:]  = Q[1] / rho
    uz[:]  = Q[2] / rho

    E = Q[3]
    p[:] = (para.gamma - 1) * (E - 0.5*rho*(ux**2 + uz**2))

# -------------------------------------------------
# Compute RHS using Finite Volume formulation
# -------------------------------------------------
def compute_rhs():
    return -compute_flux_divergence(Q)


In [ ]:
%%writefile riemann_mod.py
import numpy as np
import para_mod as para

# -------------------------------------------------
# Convert conserved to primitive locally
# -------------------------------------------------
def conserved_to_primitive(Q):
    rho = Q[0]
    u   = Q[1]/rho
    v   = Q[2]/rho
    E   = Q[3]
    p = (para.gamma - 1)*(E - 0.5*rho*(u**2+v**2))
    return rho, u, v, p

# -------------------------------------------------
# Physical flux in x-direction
# -------------------------------------------------
def flux_x(Q):
    rho, u, v, p = conserved_to_primitive(Q)
    F = np.zeros_like(Q)
    F[0] = rho*u
    F[1] = rho*u**2 + p
    F[2] = rho*u*v
    F[3] = (Q[3] + p)*u
    return F

# -------------------------------------------------
# Physical flux in z-direction
# -------------------------------------------------
def flux_z(Q):
    rho, u, v, p = conserved_to_primitive(Q)
    G = np.zeros_like(Q)
    G[0] = rho*v
    G[1] = rho*u*v
    G[2] = rho*v**2 + p
    G[3] = (Q[3] + p)*v
    return G

# -------------------------------------------------
# Rusanov flux (x-direction)
# -------------------------------------------------
def rusanov_flux_x(QL, QR):
    FL = flux_x(QL)
    FR = flux_x(QR)

    rhoL, uL, vL, pL = conserved_to_primitive(QL)
    rhoR, uR, vR, pR = conserved_to_primitive(QR)

    cL = np.sqrt(para.gamma*pL/rhoL)
    cR = np.sqrt(para.gamma*pR/rhoR)

    smax = np.maximum(np.abs(uL)+cL, np.abs(uR)+cR)

    return 0.5*(FL+FR) - 0.5*smax*(QR-QL)

# -------------------------------------------------
# Rusanov flux (z-direction)
# -------------------------------------------------
def rusanov_flux_z(QL, QR):
    GL = flux_z(QL)
    GR = flux_z(QR)

    rhoL, uL, vL, pL = conserved_to_primitive(QL)
    rhoR, uR, vR, pR = conserved_to_primitive(QR)

    cL = np.sqrt(para.gamma*pL/rhoL)
    cR = np.sqrt(para.gamma*pR/rhoR)

    smax = np.maximum(np.abs(vL)+cL, np.abs(vR)+cR)

    return 0.5*(GL+GR) - 0.5*smax*(QR-QL)


In [ ]:
%%writefile boundary_mod.py

def boundary():
    # No boundary treatment for now
    # (Periodic behavior implicit from slicing)
    pass


In [ ]:
%%writefile init_fields_mod.py
import numpy as np
import para_mod as para
import compressible_mod as cs

def init_fields():
    a0 = np.sqrt(para.gamma * para.R * para.T0)
    u0 = para.Mach0 * a0
    rho0 = para.p0 / (para.R * para.T0)

    # Initialize primitive variables
    cs.rho[:] = rho0
    cs.ux[:]  = u0
    cs.uz[:]  = 0.0
    cs.p[:]   = para.p0

    # Update conserved variables for FVM
    cs.update_conserved()


In [ ]:
%%writefile saving_mod.py
import numpy as np
import matplotlib.pyplot as plt
import compressible_mod as cs
import grid_mod as grid
import para_mod as para
import os

def print_output(t):
    print(f"t={t:.5f}  Mean U={cs.ux.mean():.3f}  Mean P={cs.p.mean():.2f}")

def save_plots():
    # Ensure primitive variables are updated
    cs.update_primitive()

    # Create output directory if it doesn't exist
    if not os.path.exists(para.output_dir):
        os.makedirs(para.output_dir)

    plt.figure()
    plt.contourf(grid.X_mesh, grid.Z_mesh, cs.ux, 30)
    plt.colorbar()
    plt.title("Velocity Contour (Modified)")
    plt.savefig(os.path.join(para.output_dir, "velocity_contour_mod.png"))
    plt.close()

    plt.figure()
    plt.contourf(grid.X_mesh, grid.Z_mesh, cs.p, 30)
    plt.colorbar()
    plt.title("Pressure Contour (Modified)")
    plt.savefig(os.path.join(para.output_dir, "pressure_contour_mod.png"))
    plt.close()


In [ ]:
%%writefile fns_mod.py
import numpy as np
import para_mod as para
import grid_mod as grid
import compressible_mod as cs
from saving_mod import print_output

def compute_dt():
    rho = cs.rho
    u   = cs.ux
    v   = cs.uz
    p   = cs.p

    c = np.sqrt(para.gamma * p / rho)
    smax = np.max(np.abs(u) + c)

    return para.CFL * grid.dx / smax


def time_advance_euler():
    t = para.tinit

    while t < para.tfinal:

        # Ensure conserved variables are correct
        cs.update_conserved()

        # Compute stable timestep
        dt = compute_dt()

        # Compute RHS using FVM divergence
        rhs = cs.compute_rhs()

        # Update ALL conserved variables (4 equations)
        cs.Q += dt * rhs

        # Update primitive variables
        cs.update_primitive()

        # Output
        if int(t/para.t_p) != int((t+dt)/para.t_p):
            print_output(t)

        t += dt


In [ ]:
%%writefile main_mod.py
import time
from init_fields_mod import init_fields
from fns_mod import time_advance_euler
from saving_mod import save_plots
import compressible_mod as cs

ti = time.time()

# Initialize fields
init_fields()

# Time advancement
time_advance_euler()

# Save final plots
save_plots()

tf = time.time()

print("\nFinal Results (Modified Solver):")
print("Mean Velocity =", cs.ux.mean())
print("Mean Pressure =", cs.p.mean())
print("Total time =", tf-ti, "seconds")


In [ ]:
!python main_mod.py